In [1]:
"""
Kaggle Notebook 용 딥페이크 탐지 모델 학습 코드 (PyTorch)
Kaggle의 셀(Cell) 단위로 복사해서 실행하시면 됩니다.

사전 준비: Kaggle Notebook 첫 번째 셀에서 아래 명령어를 실행해주세요.
"""
!pip install facenet-pytorch --no-deps




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.3 MB/s eta 0:00:0000:0100:01


In [2]:

import os
import cv2
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm import tqdm
from PIL import Image

try:
    from facenet_pytorch import MTCNN
except ImportError:
    print("MTCNN 라이브러리가 없습니다. '!pip install facenet-pytorch' 를 먼저 실행해주세요.")

# ==========================================
# 1. 설정 및 하이퍼파라미터
# ==========================================
# Kaggle에 업로드한 데이터셋 경로에 맞게 수정해야 합니다.
# (예: /kaggle/input/deepfake-dataset/New_sample (2)/원천데이터/)
BASE_DIR = "/kaggle/input/datasets/seongbeenkim/deepfakesample/New_sample (2)/원천데이터"  # 로컬 테스트용 경로
REAL_DIR = os.path.join(BASE_DIR, "train_원본")
FAKE_DIR = os.path.join(BASE_DIR, "train_변조")

# 하이퍼파라미터
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {DEVICE}")

# ==========================================
# 2. 얼굴 추출 및 데이터셋 클래스
# ==========================================
class DeepfakeVideoDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None, frames_per_video=5):
        self.real_dir = real_dir
        self.fake_dir = fake_dir
        self.transform = transform
        self.frames_per_video = frames_per_video # 영상 1개당 추출할 프레임 수
        
        # MTCNN: 영상에서 얼굴만 잘라내는 인공지능 모델
        self.mtcnn = MTCNN(keep_all=False, device=DEVICE)
        
        self.samples = [] # (video_path, label) 리스트. 0: 원본, 1: 딥페이크
        
        # 원본 영상 로드 (Label 0)
        real_videos = glob.glob(os.path.join(real_dir, '**', '*.mp4'), recursive=True)
        for v in real_videos:
            self.samples.append((v, 0))
            
        # 변조 영상 로드 (Label 1)
        fake_videos = glob.glob(os.path.join(fake_dir, '**', '*.mp4'), recursive=True)
        for v in fake_videos:
            self.samples.append((v, 1))
            
        print(f"원본 영상 개수: {len(real_videos)}, 변조 영상 개수: {len(fake_videos)}")

    def __len__(self):
        return len(self.samples)

    def extract_faces(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        faces = []
        
        if total_frames == 0:
            return faces

        # 영상을 일정한 간격으로 추출 (예: 5장)
        interval = max(1, total_frames // self.frames_per_video)
        
        for i in range(self.frames_per_video):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
            ret, frame = cap.read()
            if not ret: continue
                
            # OpenCV는 BGR을 쓰므로 RGB로 변환
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(frame_rgb)
            
            # MTCNN으로 얼굴만 잘라냄
            face = self.mtcnn(img_pil)
            if face is not None:
                faces.append(face)
                
        cap.release()
        return faces # [Tensor, Tensor, ...]

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        
        # 영상에서 얼굴 텐서들 추출
        faces = self.extract_faces(video_path)
        
        # 얼굴을 찾지 못한 경우 (예외 처리: 검은 화면 반환)
        if len(faces) == 0:
            faces = [torch.zeros((3, 224, 224))]
            
        # 여러 얼굴 중 무작위로 1개 선택해서 학습에 사용
        import random
        selected_face = random.choice(faces)
        
        return selected_face, label

# ==========================================
# 3. 모델 정의 (EfficientNet-B0)
# ==========================================
def get_model():
    # ImageNet으로 사전학습된 EfficientNet 로드
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    
    # 마지막 분류기(Classifier)를 이진 분류(0: Real, 1: Fake)로 수정
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid() # 0~1 사이의 확률값 출력
    )
    return model.to(DEVICE)

# ==========================================
# 4. 학습 루프 (Training Loop)
# ==========================================
def train_model():
    print("데이터셋 준비 중...")
    train_dataset = DeepfakeVideoDataset(REAL_DIR, FAKE_DIR, frames_per_video=3)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    
    model = get_model()
    criterion = nn.BCELoss() # 이진 분류 손실 함수
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    print("학습 시작!")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        corrects = 0
        total = 0
        
        # TQDM으로 진행률 표시
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for inputs, labels in pbar:
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE) # [Batch, 1] 형태로 변환
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
            # 0.5 이상이면 Fake(1), 미만이면 Real(0)
            preds = (outputs >= 0.5).float()
            corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
            
            pbar.set_postfix({'Loss': loss.item()})
            
        epoch_loss = running_loss / total
        epoch_acc = corrects.double() / total
        print(f"Epoch {epoch+1} 완료 - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
        
    # 학습 완료 후 가중치 저장
    save_path = "deepfake_model_weights.pth"
    torch.save(model.state_dict(), save_path)
    print(f"가중치 저장 완료: {save_path}")
    print("수고하셨습니다! 이 .pth 파일을 다운로드해서 로컬로 가져오시면 됩니다.")

if __name__ == "__main__":
    # Kaggle 노트북에서 실행할 경우 아래 주석을 풀고 실행하세요.
    train_model()
    pass

사용 장치: cuda
데이터셋 준비 중...
원본 영상 개수: 44, 변조 영상 개수: 44
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 136MB/s] 


학습 시작!


Epoch 1/5: 100%|██████████| 6/6 [01:43<00:00, 17.30s/it, Loss=0.435]


Epoch 1 완료 - Loss: 0.5871, Accuracy: 0.7386


Epoch 2/5: 100%|██████████| 6/6 [01:39<00:00, 16.57s/it, Loss=0.302]


... (3 outputs left out)

Epoch 4/5: 100%|██████████| 6/6 [01:34<00:00, 15.83s/it, Loss=0.146]


Epoch 4 완료 - Loss: 0.2002, Accuracy: 1.0000


Epoch 5/5: 100%|██████████| 6/6 [01:34<00:00, 15.78s/it, Loss=0.0791]

Epoch 5 완료 - Loss: 0.1287, Accuracy: 1.0000
가중치 저장 완료: deepfake_model_weights.pth
수고하셨습니다! 이 .pth 파일을 다운로드해서 로컬로 가져오시면 됩니다.
